In [1]:
import sys
from pathlib import Path

project_root = next(
    (
        path
        for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (path / "src" / "data_collection" / "consts.py").is_file()
    ),
    None,
)
if project_root is None:
    raise RuntimeError("Could not locate project root containing 'src' directory.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import torch
import psycopg2
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

from src.data_collection.consts import DB_PARAMS
from src.data_collection.model_driver.model_driver_class import ModelDriver
from src.data_collection.logging_config import logger

/home/maxim-shibanov/anaconda3/envs/vllm_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
verolizer = {
    'positive': ['buy'],
    'negative': ['sell'],
}

model_name = "mistralai/Mistral-7B-v0.1"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map = "auto",
    torch_dtype=torch.bfloat16,
    max_memory={0: "13GiB", "cpu": "64GiB"},  
)

model_driver = ModelDriver(model_name=model_name, model=model, verbolizer=verolizer)

Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.26it/s]
2026-03-26 17:53:40,858 [WARNING] Some parameters are on the meta device because they were offloaded to the cpu.


In [3]:
report = model_driver.get_random_reports()[0]

In [4]:
print(report[:500])





Document


 UNITED STATESSECURITIES AND EXCHANGE COMMISSIONWashington, DC 20549 FORM 10-Q QUARTERLY   REPORT   PURSUANT   TO   SECTION   13   OR   15(d)   OF   THE   SECURITIES EXCHANGE ACT OF 1934For the quarterly period ended September 30, 2018 orTRANSITION   REPORT   PURSUANT   TO   SECTION   13   OR   15(d)   OF   THE  SECURITIES EXCHANGE ACT OF 1934For the transition period from              to             Commission file number 001-09718The PNC Financial Services Group, Inc.(Exact name


In [5]:
model_driver.infer_report_type(text=report)

2026-03-26 17:53:44,938 [WARNING] Report type scores sum to 0. Returning zeros.


[('annual', 0.455078125), ('quarter', 0.40234375), ('Annual', 0.01214599609375), ('Quarter', 0.0078125), ('SEC', 0.003936767578125), ('AN', 0.003265380859375), ('it', 0.0025482177734375), ('annually', 0.0025482177734375), ('an', 0.00131988525390625), ('Qu', 0.00106048583984375), ('as', 0.000934600830078125), ('wrong', 0.00080108642578125), ('ann', 0.0007781982421875), ('qu', 0.000751495361328125), ('SE', 0.000751495361328125), ('one', 0.000568389892578125), ('incorrect', 0.00054931640625), ('NO', 0.0005340576171875), ('semi', 0.000518798828125), ('either', 0.0004711151123046875)]


{'10-K': 0.0, '10-Q': 0.0}